# Visualização de Focus de Modelos

### Notebook para Carregar e Visualizar, Via Mapas de Calor, o Foco dos Modelos nas Imagens

In [22]:
import torch
import torchvision
from torchvision import transforms

from tensorflow.keras.preprocessing import image

import sys
import os

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import tqdm

import cv2
import tensorflow as tf

project_root = os.path.dirname(os.path.dirname(os.getcwd()))
sys.path.append(project_root)

from utils.models_to_pkl import load_model

In [23]:
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

Using device: cuda


In [24]:
model_path = os.path.join(project_root, "models", "modelo_binario_co2wounds")
model = load_model(model_path)

✅ Modelo carregado de: /home/alyssandro/Documents/Github/leprosy_classification_models-TCC/models/modelo_binario_co2wounds.keras


## Esse trecho serve apenas para oos modelos treinados com os pesos do imagenet - (1, 224, 224, 3)

In [25]:
def get_img_array(img_path, size=(224, 224)):
    # Carrega e redimensiona
    img = image.load_img(img_path, target_size=size)
    # Converte para array
    array = image.img_to_array(img)
    # Expande dimensão para batch (1, 224, 224, 3)
    array = np.expand_dims(array, axis=0)
    # Normaliza (como na ResNet50)
    array = tf.keras.applications.resnet50.preprocess_input(array)
    return array

def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    """Gera o mapa de calor Grad-CAM."""
    grad_model = tf.keras.models.Model(
        [model.inputs], 
        [model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    # Gradiente da classe com relação aos mapas de ativação
    grads = tape.gradient(class_channel, conv_outputs)

    # Média espacial dos gradientes
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    # Normaliza entre 0 e 1
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()

def display_gradcam(img_path, heatmap, alpha=0.5, gamma=2.0, percentile=80):
    """
    Grad-CAM tipo 'fogo' mais destacado: azul -> vermelho intenso.
    percentile: define limite máximo do heatmap para aumentar contraste
    gamma: intensifica contraste
    alpha: transparência do heatmap sobre a imagem original
    """
    import cv2
    import numpy as np
    import matplotlib.pyplot as plt

    # Carrega imagem
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Redimensiona heatmap
    heatmap = cv2.resize(heatmap, (img.shape[1], img.shape[0]))

    # Normalização baseada em percentil para maior contraste
    max_val = np.percentile(heatmap, percentile)
    heatmap = np.clip(heatmap / (max_val + 1e-8), 0, 1)
    
    # Aplica gamma para realçar regiões importantes
    heatmap = np.power(heatmap, gamma)
    
    # Converte para 0-255
    heatmap = np.uint8(255 * heatmap)
    
    # Colormap JET (azul -> vermelho)
    heatmap_colored = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    
    # Sobreposição
    superimposed_img = cv2.addWeighted(heatmap_colored, alpha, img, 1 - alpha, 0)
    
    # Mostra imagens
    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1)
    plt.title("Imagem Original")
    plt.imshow(img)
    plt.axis('off')
    
    plt.subplot(1, 2, 2)
    plt.title("Grad-CAM Destacado")
    plt.imshow(superimposed_img)
    plt.axis('off')
    
    plt.show()

In [26]:
for i, layer in enumerate(model.layers): # Achando o nome da última camada convolucional
    try:
        print(f"{i}: {layer.name} -> {layer.output_shape}")
    except AttributeError:
        print(f"{i}: {layer.name} -> (sem output_shape)")


0: input_layer -> (sem output_shape)
1: conv1_pad -> (sem output_shape)
2: conv1_conv -> (sem output_shape)
3: conv1_bn -> (sem output_shape)
4: conv1_relu -> (sem output_shape)
5: pool1_pad -> (sem output_shape)
6: pool1_pool -> (sem output_shape)
7: conv2_block1_1_conv -> (sem output_shape)
8: conv2_block1_1_bn -> (sem output_shape)
9: conv2_block1_1_relu -> (sem output_shape)
10: conv2_block1_2_conv -> (sem output_shape)
11: conv2_block1_2_bn -> (sem output_shape)
12: conv2_block1_2_relu -> (sem output_shape)
13: conv2_block1_0_conv -> (sem output_shape)
14: conv2_block1_3_conv -> (sem output_shape)
15: conv2_block1_0_bn -> (sem output_shape)
16: conv2_block1_3_bn -> (sem output_shape)
17: conv2_block1_add -> (sem output_shape)
18: conv2_block1_out -> (sem output_shape)
19: conv2_block2_1_conv -> (sem output_shape)
20: conv2_block2_1_bn -> (sem output_shape)
21: conv2_block2_1_relu -> (sem output_shape)
22: conv2_block2_2_conv -> (sem output_shape)
23: conv2_block2_2_bn -> (sem outp

In [ ]:
for i in os.listdir(project_root + "/data/CO2Wounds-V2/raw/train_images_binary/train/leprosy/"):
    img_path = project_root + "/data/CO2Wounds-V2/raw/train_images_binary/train/leprosy/" + i
    

    img_array = get_img_array(img_path, size=(224, 224))
    heatmap = make_gradcam_heatmap(img_array, model, last_conv_layer_name="conv5_block3_out")
    display_gradcam(img_path, heatmap)

In [ ]:
prediction = model.predict(img_array)

# Faz a predição
prediction = model.predict(img_array)

# Como é um modelo binário com saída sigmoid, teremos um valor entre 0 e 1
print(f"Probabilidade predita: {prediction[0][0]:.4f}")

# Define o limiar de decisão (threshold)
threshold = 0.5
classe_predita = "Classe 1" if prediction[0][0] >= threshold else "Classe 0"

print(f"Classe predita: {classe_predita}") # 0 lepra, 1 outros

1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
Probabilidade predita: 0.0000
Classe predita: Classe 0
